In [1]:
##### Converts raw raster on population into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # people per km2

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats
from rasterio.transform import xy

In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# country geographies 
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0")

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")
geo_area = pd.read_csv(f"{cd}/Data/Clean/Geographies/subnational_total_area.csv")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# total population
total_pop = f"{cd}/Data/Raw/Predictors/Population_Bondarenko/global_agesex_structures_2020_CN_1km_R2025A_UA_v1/total_pop_2020.tif"

# WB population density 
WB_pop_density = pd.read_csv(f"{cd}/Data/Raw/Predictors/Population_Bondarenko/WorldBank_pop_density.csv", skiprows=4)
names = pd.read_csv(f'{cd}/Data/Correspondence_tables/country_names.csv', encoding="cp1252")

# save paths
pixels_pop_resample = f"{cd}/Data/Raw/Predictors/Population_Bondarenko/global_agesex_structures_2020_CN_1km_R2025A_UA_v1/pop_total_resample_2020.tif"
pixels = f"{cd}/Data/Clean/Predictors/Rasters/pop_density.tif"
vectors = f"{cd}/Data/Clean/Predictors/Vectors/pop_density.csv"
vectors_country = f"{cd}/Data/Clean/Predictors/Vectors/Country_averages/pop_density.csv"

KeyboardInterrupt: 

In [7]:
#### Run resampling for pixel scale 

### PREP 
# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()


### RESAMPLE 
def resample_to_ref(src_path):
    dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.sum,
            dst_nodata=np.nan,
        )
    dst_array[dst_array == -9999] = np.nan
    return dst_array

pop  = resample_to_ref(total_pop)

# save 
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = pop.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels_pop_resample, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

In [11]:
#### create population density raster

# Read total population raster
with rasterio.open(pixels_pop_resample) as src:
    pop = src.read(1).astype(np.float32)
    nodata = src.nodata
    transform = src.transform
    meta = src.meta.copy()
    height, width = src.shape

# Mask nodata
pop[pop == nodata] = np.nan

# Compute pixel area in km² for each row (varies with latitude)
# Get latitude of each row's pixel centre
rows = np.arange(height)
cols = np.zeros(height, dtype=int)  # column doesn't matter for N-S pixel size
_, lats = xy(transform, rows, cols)  # centre latitudes for each row
lats_rad = np.deg2rad(np.array(lats))

# Pixel dimensions in degrees
d_lon = abs(transform.a)  # width in degrees
d_lat = abs(transform.e)  # height in degrees

# Earth radius in km
R = 6371.0

# N-S extent in km (constant across longitude)
dy_km = R * np.deg2rad(d_lat)

# E-W extent in km (shrinks with cos(latitude))
dx_km = R * np.cos(lats_rad) * np.deg2rad(d_lon)

# Pixel area in km² — shape (height,), broadcast to (height, width)
pixel_area_km2 = (dy_km * dx_km)[:, np.newaxis] * np.ones((1, width))

# Population density
with np.errstate(invalid="ignore", divide="ignore"):
    pop_density = np.where(
        ~np.isnan(pop),
        pop / pixel_area_km2,
        np.nan
    )

# Save
meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")
out_arr = pop_density.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels, "w", **meta) as dst:
    dst.write(out_arr, 1)

Done — density range: 0.00 – 53189.23 people/km²


In [3]:
#### Calculate total population in vector (sub-national)

### Set-up 
# repoject GDF to match raster 
gdf_proj = total_geo.to_crs("EPSG:4326")

result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE
# sum irrigation and cropland in each vector
zonal_pop  = rasterstats.zonal_stats(gdf_proj, pixels_pop_resample, stats="sum", nodata=-9999)

result["total_population"] = [z["sum"] if z["sum"] is not None else np.nan for z in zonal_pop]
result["total_population"] = result["total_population"].fillna(0)

In [4]:
### Calculate population density (sub-national)

# merge with area
result = result.merge(geo_area, on='PROJ_ID', how='outer')

# calculate
result['pop_density_people_per_km2'] = result['total_population'] / result['area_km2']

# drop vars
result = result[['PROJ_ID', 'pop_density_people_per_km2']]

### save
result.to_csv(vectors, index=False)

In [5]:
#### Clean WB country average data

### clean WB data
WB_pop_density['pop_density_people_per_km2'] = WB_pop_density['2016']
WB_pop_density['WB_code'] = WB_pop_density['Country Code']
WB_pop_density = WB_pop_density[['WB_code', 'pop_density_people_per_km2']]

# merge with country codes
WB_pop_density = WB_pop_density.merge(names[['SHP_code', 'WB_code']], on='WB_code', how='right')
WB_pop_density['GID_0'] = WB_pop_density['SHP_code']
WB_pop_density = WB_pop_density[['GID_0', 'pop_density_people_per_km2']]

### SAVE
WB_pop_density.to_csv(vectors_country, index=False)